# Khảo sát dữ liệu ViPRA
Chạy từ trên xuống. Notebook chỉ đọc dữ liệu gốc.

Thư viện: `pandas`, `matplotlib`, `transformers`, `sentencepiece`, `jinja2`.
Nếu thiếu, chạy trong một ô: `%pip install pandas matplotlib transformers sentencepiece jinja2`.


## 1. Đọc dữ liệu
Đổi `DATA_DIR` khi chuyển dataset. Tokenizer dưới đây khớp backbone Qwen2.5-Math của baseline.
Chỉ tải tokenizer, không tải trọng số LLM.


In [ ]:
%pip install pandas matplotlib transformers sentencepiece jinja2    

In [ ]:
from pathlib import Path
import json, re, unicodedata
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

%matplotlib inline

DATA_DIR = Path("/Users/nho/Documents/NCKH /VLSP Dataset/")
FILES = {"train": "train_dataset.csv", "dev": "dev_dataset.csv"}
TOKENIZER_ID = "Qwen/Qwen2.5-Math-7B-Instruct"
TOKENIZER_REVISION = "ef9926d75ab1d54532f6a30dd5e760355eb9aa4d"
LIMITS = [1024, 2048, 4096]
COLS = ["id", "question", "cot", "answer", "unit"]

frames = {}
for split, filename in FILES.items():
    df = pd.read_csv(DATA_DIR / filename, dtype=str, keep_default_na=False)
    if missing := set(COLS) - set(df.columns):
        raise ValueError(f"{filename} thiếu cột: {sorted(missing)}")
    frames[split] = df[COLS].copy()
data = pd.concat([df.assign(split=s) for s, df in frames.items()], ignore_index=True)
display(data.groupby("split").size().rename("Số mẫu").to_frame())



## 2. Trường rỗng và ID



In [ ]:
empty = data[COLS].apply(lambda s: s.str.strip().eq(""))
print("Số ô rỗng")
display(empty.groupby(data["split"]).sum().T.rename_axis("Cột rỗng"))
print("kiểm tra id")
display(data.groupby("split")["id"].agg(total="size", unique="nunique"))
escaped = data[["question", "cot"]].apply(lambda s: s.str.contains("\\n", regex=False))
#dòng bị lỗi đã phát hiện
print("có 1 dòng bi lỗi không có đáp án:")
display(data.loc[
    data["split"].eq("train") & data["answer"].str.strip().eq("")
])

## 3. Trùng và nhãn khác nhau



In [ ]:
data["qkey"] = data["question"].map(lambda x: " ".join(unicodedata.normalize("NFC", x).split()))
data["label"] = data["answer"].str.strip() + " | " + data["unit"].str.strip()
train_keys = set(data.loc[data["split"].eq("train") & data["qkey"].ne(""), "qkey"])
data["overlap"] = data["split"].eq("dev") & data["qkey"].ne("") & data["qkey"].isin(train_keys)
data["conflict"] = data["qkey"].ne("") & data.groupby("qkey")["label"].transform("nunique").gt(1)

quality = data.assign(
    duplicate_row=data.duplicated(["split"] + COLS, keep=False),
    duplicate_id=data["id"].str.strip().ne("") & data.duplicated(["split", "id"], keep=False),
    duplicate_question=data["qkey"].ne("") & data.duplicated(["split", "qkey"], keep=False),
)
display(quality.groupby("split")[["duplicate_row", "duplicate_id", "duplicate_question", "overlap", "conflict"]].sum())


## 4. Độ dài ký tự và token
Code dùng tokenizer của Qwen/Qwen2.5-Math-7B-Instruct

In [ ]:
def length_summary(lengths):
    return (lengths.groupby(["split", "field"])["length"]
            .agg(min="min", median="median", mean="mean", p95=lambda s: s.quantile(.95), max="max")
            .round(1))

chars = data[["split", "question", "cot"]].copy()
chars[["question", "cot"]] = chars[["question", "cot"]].apply(lambda s: s.str.len())
display(length_summary(chars.melt("split", var_name="field", value_name="length")))


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, revision=TOKENIZER_REVISION)
if not tokenizer.chat_template:
    raise ValueError("Tokenizer chưa có chat template; chọn tokenizer của mô hình chat/instruct.")
SYSTEM = "Giải bài toán vật lý. Trả JSON gồm answer (không kèm đơn vị), unit và explanation."

def sft_text(row):
    target = {"answer": row.answer, "unit": row.unit,
              "explanation": row.cot.replace("\\n", "\n")}
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": row.question},
                {"role": "assistant", "content": json.dumps(target, ensure_ascii=False)}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

token_lengths = pd.DataFrame({"split": data["split"]})
texts = {"question": data["question"].tolist(), "cot": data["cot"].str.replace("\\n", "\n", regex=False).tolist(),
         "sft": [sft_text(row) for row in data.itertuples()]}
for field, values in texts.items():
    token_lengths[field] = [len(ids) for ids in tokenizer(values, add_special_tokens=False)["input_ids"]]
display(length_summary(token_lengths.melt("split", var_name="field", value_name="length")))
display(token_lengths.groupby("split")["sft"].sum().rename("Tổng token SFT").to_frame())
display(pd.DataFrame({n: token_lengths["sft"].gt(n).groupby(data["split"]).mean().mul(100)
                      for n in LIMITS}).rename_axis(columns="Ngưỡng: % mẫu vượt"))
fig, ax = plt.subplots(figsize=(7, 3))
for split, group in token_lengths.groupby("split"):
    ax.hist(group["sft"], bins=25, alpha=.55, label=split)
ax.set(xlabel="Tokens / SFT sample", ylabel="Samples")
ax.legend()
plt.show()


## 5. Dạng đáp án và đơn vị
Phân loại bằng quy tắc đơn giản, cần đọc lại trường hợp đặc biệt. Giữ nguyên đơn vị gốc;
`-`, `—`, `μ`, `µ`, `Ω` được thống kê để xử lý sau, không tự sửa ở notebook này.


In [ ]:
NUMBER = r"[+-]?(?:\d+(?:[.,]\d*)?|[.,]\d+)(?:[eE][+-]?\d+)?"
SCIENCE = NUMBER + r"\s*(?:\*|×|x|\\times)\s*10(?:\s*\^\s*\{?[+-]?\d+\}?|[⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺]+)"

def answer_type(text):
    text = text.strip()
    if not text: return "Rỗng"
    if ";" in text: return "Nhiều phần"
    if re.fullmatch(NUMBER, text) or re.fullmatch(SCIENCE, text): return "Số"
    if re.search(r"[=+*/^()√]|\\(?:frac|sqrt)|\b(?:sqrt|sin|cos|pi)\b|\d", text): return "Biểu thức/khác"
    return "Văn bản/ký hiệu"

data["answer_type"] = data["answer"].map(answer_type)
types = pd.crosstab(data["answer_type"], data["split"])
display(types)
display(types.div(types.sum()).mul(100).round(1).rename_axis("Tỷ lệ (%)"))
units = pd.crosstab(data["unit"].replace("", "(rỗng)"), data["split"])
display(units.loc[units.sum(axis=1).sort_values(ascending=False).index])


## 6. Chủ đề sơ bộ
Đếm theo từ khóa, **một câu có thể thuộc nhiều nhóm**. Đây không phải nhãn chủ đề chính thức.


In [ ]:
patterns = {
    "Coulomb / điện tích": r"coulomb|điện tích|lực điện",
    "Điện trường / điện thế": r"điện trường|điện thế|điện thông",
    "Tụ điện": r"tụ điện|\btụ\b|điện dung",
    "Từ trường / cảm ứng": r"từ trường|từ thông|cảm ứng|solenoid|ống dây|cuộn dây",
    "Mạch điện": r"mạch|xoay chiều|cộng hưởng|cảm kháng|dung kháng",
    "Sai số / đo lường": r"sai số|độ không đảm bảo|phép đo",
}
topics = pd.DataFrame({name: data["question"].str.contains(p, case=False, regex=True)
                       for name, p in patterns.items()})
topics["Chưa khớp từ khóa"] = ~topics.any(axis=1)
display(topics.groupby(data["split"]).sum().T)
display(topics.groupby(data["split"]).mean().mul(100).round(1).T.rename_axis("Tỷ lệ (%)"))
